# Formula-SWCT Generalization on FGVC-Aircraft

This notebook evaluates the final Formula-SWCT method on the FGVC-Aircraft fine-grained image-classification dataset using an ImageNet-pretrained ResNet-18.

FGVC-Aircraft introduces a challenging fine-grained recognition setting that differs substantially from the three Formula-SWCT calibration datasets:

- Beans
- DTD
- Flowers102

The evaluation follows the final predict-first, train-once protocol:

1. Collect stage statistics from the target training split using the frozen pretrained network.
2. Predict four stage-specific beta values using the frozen Formula-SWCT predictor.
3. Freeze the predicted beta values.
4. Extract fixed train, validation, and test features once.
5. Train one downstream linear classifier.
6. Select the classifier checkpoint using validation accuracy.
7. Evaluate once on the test split.

No FGVC-Aircraft-specific beta sweep or beta optimization is performed.

## Environment Setup

This notebook is designed to run independently in Google Colab.

The repository is cloned or updated, the `stage-wise-ct` branch is selected, and the research dependencies are installed.

After dependency installation, restart the Colab runtime before continuing.

In [1]:
!nvidia-smi

%cd /content

import os

if not os.path.exists("/content/curvature-tuning-research"):
    !git clone -b stage-wise-ct https://github.com/Syed1611/curvature-tuning-research.git
else:
    print("Repository already exists; skipping clone.")

%cd /content/curvature-tuning-research
!git checkout stage-wise-ct
!git pull

%cd /content/curvature-tuning-research/src/curvature-tuning
!git branch --show-current

Tue Sep 29 20:25:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 40.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 66.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 98.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 107.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 95.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Runtime Restart Required

Restart the Colab runtime after dependency installation.

After restarting, do not rerun the installation cell. Continue from the next cell.

In [1]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning


In [2]:
import torch
import datasets
import numpy
import pandas
import sklearn
import tqdm
import loguru

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("datasets:", datasets.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("tqdm:", tqdm.__version__)
print("loguru:", loguru.__version__)

Torch: 2.6.0+cu124
CUDA: True
GPU: Tesla T4
datasets: 3.4.1
NumPy: 1.26.3
Pandas: 2.2.3
sklearn: 1.5.2
tqdm: 4.66.5
loguru: 0.7.2


## FGVC-Aircraft Dataset Setup

The FGVC-Aircraft train, validation, and test loaders are constructed using the project's standard transfer-learning pipeline.

This cell verifies that the dataset can be loaded correctly before running the evaluation methods.

In [3]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_fgvc-aircraft",
    train_batch_size=32,
    test_batch_size=64,
    seed=42
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Label min:", labels.min().item())
print("Label max:", labels.max().item())

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

fgvc-aircraft.py: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Train samples: 3334
Validation samples: 3333
Test samples: 3333
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Label min: 5
Label max: 97


# Baseline and Original S-CT Reference

The original Baseline and Single-Parameter Curvature Tuning methods are included as reference methods.

The Baseline uses the original frozen ImageNet-pretrained ResNet-18 representation.

S-CT searches for one global beta value shared across all curvature-modified activations using the original validation-based beta-selection procedure.

The original Aircraft experiment did not complete this comparison, so this section is retained for later execution to obtain a fair reference against Formula-SWCT.

In [4]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

2026-09-29 21:25:20.887 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_fgvc-aircraft_resnet18_seed42.log
2026-09-29 21:25:20.894 | INFO     | __main__:main:58 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 172MB/s]
2026-09-29 21:25:24.052 | INFO     | __main__:main:85 - Testing baseline...
2026-09-29 21:25:24.073 | INFO     | __main__:main:88 - Number of trainable parameters: 51300
2026-09-29 21:25:24.073 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-29 21:28:36.233 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.636715, Accuracy: 0.00%
2026-09-29 21:28:36.511 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 4.643287, Accuracy: 2.17%
2026-09-29 21:28:36.759 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.321471, Val Accuracy: 6.03%
2026-09-29 21:28:36.760 | INFO    

# Fit the Frozen Formula-SWCT Predictor

Because this notebook is designed to run independently in a fresh Google Colab runtime, the Formula-SWCT predictor is reconstructed before evaluating FGVC-Aircraft.

The formula is fitted using only the three fixed development/calibration datasets:

| Dataset | Stage 1 | Stage 2 | Stage 3 | Stage 4 |
|---|---:|---:|---:|---:|
| Beans | 0.85 | 0.85 | 0.85 | 0.80 |
| DTD | 0.95 | 0.95 | 0.99 | 0.99 |
| Flowers102 | 0.99 | 0.95 | 0.99 | 0.90 |

For each calibration dataset and seed, the frozen ImageNet-pretrained ResNet-18 is used to collect five stage descriptors:

1. log activation variance,
2. activation sparsity,
3. log mean absolute activation,
4. log class-separation statistic,
5. normalized stage depth.

These observations are used to fit the fixed ridge-regression Formula-SWCT predictor with:

**lambda = 0.10**

This fitting step is performed before loading FGVC-Aircraft for Formula-SWCT evaluation. FGVC-Aircraft does not modify the fitted formula.

In [5]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!python formula_swct.py \
    --mode fit \
    --model resnet18 \
    --pretrained_ds imagenet \
    --train_bs 32 \
    --test_bs 64

/content/curvature-tuning-research/src/curvature-tuning
FORMULA-SWCT: FIT DIRECT BETA FORMULA
No beta sweep and no downstream training are performed here.

data/train-00000-of-00001.parquet: 100% 144M/144M [00:01<00:00, 73.6MB/s] 
data/validation-00000-of-00001.parquet: 100% 18.5M/18.5M [00:00<00:00, 19.2MB/s]
data/test-00000-of-00001.parquet: 100% 17.7M/17.7M [00:00<00:00, 27.0MB/s]
Generating train split: 1034 examples [00:00, 1796.76 examples/s]
Generating validation split: 133 examples [00:00, 3552.16 examples/s]
Generating test split: 128 examples [00:00, 3398.78 examples/s]



100% 625M/625M [00:34<00:00, 18.2MB/s] 



100% 345M/345M [00:20<00:00, 16.5MB/s] 
100% 502/502 [00:00<00:00, 1.95MB/s]
100% 15.0k/15.0k [00:00<00:00, 35.3MB/s]



FITTED FORMULA
beta_hat = clip(b0 + sum(w_j * z_j), 0.7, 0.99)
intercept: 0.92166667
            log_variance: +0.05225792
                sparsity: -0.03829733
            log_mean_abs: -0.08039330
    log_class_separation: +0.04280035
         

# Final Formula-SWCT Evaluation

The frozen Formula-SWCT predictor is now applied to FGVC-Aircraft.

The predictor first measures the stage statistics of the frozen ImageNet-pretrained ResNet-18 using only the Aircraft training split.

The four stage-specific beta values are then predicted before downstream classifier training.

After prediction:

- all four beta values are frozen,
- train/validation/test features are extracted once,
- one downstream linear classifier is trained,
- validation accuracy selects the classifier checkpoint,
- the selected classifier is evaluated once on the test split.

No Aircraft-specific beta sweep or beta optimization is performed.

In [6]:
!python formula_swct.py \
    --mode run \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --train_bs 32 \
    --test_bs 64 \
    --epochs 20

FORMULA-SWCT: PREDICT FIRST, TRAIN ONCE
Dataset: fgvc-aircraft
Seed: 42
No beta sweep. No beta optimization. No 0.78 anchor.

Predicted betas BEFORE training:
Stage 1: 0.899803
Stage 2: 0.887323
Stage 3: 0.889689
Stage 4: 0.863769
Predicted beta vector: [0.899803, 0.887323, 0.889689, 0.863769]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
Loaded fixed betas: [0.899803, 0.887323, 0.889689, 0.863769]

Extracting fixed Formula-SWCT features once...
Training ONE downstream classifier...
Epoch 01 | val_loss=4.249081 | val_acc=8.58%
Epoch 02 | val_loss=3.686834 | val_acc=17.34%
Epoch 03 | val_loss=3.338110 | val_acc=23.40%
Epoch 04 | val_loss=3.135462 | val_acc=26.70%
Epoch 05 | val_loss=3.038165 | val_acc=27.15%
Epoch 06 | val_loss=2.913319 | val_acc=30.54%
Epoch 07 | val_loss=2.844654 | val_acc=31.14%
Epoch 08 | val_loss=2.790001 | val_acc=32.73%
Epoch 09 | val_loss=2.732918 | val_acc=33.27%
Epoch 10 | val_loss=2.708958 | val_acc=34.32%
Epoch 11 | val_loss=2.602686 | val_acc=36.96%
Epoch 12 | va

In [7]:
!python formula_swct.py \
    --mode run \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --train_bs 32 \
    --test_bs 64 \
    --epochs 30

FORMULA-SWCT: PREDICT FIRST, TRAIN ONCE
Dataset: fgvc-aircraft
Seed: 42
No beta sweep. No beta optimization. No 0.78 anchor.

Predicted betas BEFORE training:
Stage 1: 0.899803
Stage 2: 0.887323
Stage 3: 0.889689
Stage 4: 0.863769
Predicted beta vector: [0.899803, 0.887323, 0.889689, 0.863769]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
Loaded fixed betas: [0.899803, 0.887323, 0.889689, 0.863769]

Extracting fixed Formula-SWCT features once...
Training ONE downstream classifier...
Epoch 01 | val_loss=4.249081 | val_acc=8.58%
Epoch 02 | val_loss=3.686834 | val_acc=17.34%
Epoch 03 | val_loss=3.338110 | val_acc=23.40%
Epoch 04 | val_loss=3.135462 | val_acc=26.70%
Epoch 05 | val_loss=3.038165 | val_acc=27.15%
Epoch 06 | val_loss=2.913319 | val_acc=30.54%
Epoch 07 | val_loss=2.844654 | val_acc=31.14%
Epoch 08 | val_loss=2.790001 | val_acc=32.73%
Epoch 09 | val_loss=2.732918 | val_acc=33.27%
Epoch 10 | val_loss=2.708958 | val_acc=34.32%
Epoch 11 | val_loss=2.602686 | val_acc=36.96%
Epoch 12 | va

## Previously Completed Formula-SWCT Run

A Formula-SWCT FGVC-Aircraft seed-42 experiment was completed before the original notebook data was lost.

The recorded result was:

- Predicted beta vector: **[0.899803, 0.887323, 0.889689, 0.863769]**
- Best validation accuracy: **37.47%**
- Best validation epoch: **19**
- Test accuracy: **38.10%**

This result was obtained using the frozen Formula-SWCT predictor before an Aircraft S-CT comparison was completed.

The experiment will be reproduced using the execution cell above so that the result and generated JSON file are restored in this notebook.

# FGVC-Aircraft Results Summary

The Formula-SWCT seed-42 result currently available is:

| Method | Beta Selection | Test Accuracy |
|---|---|---:|
| Formula-SWCT | Direct four-stage prediction | **38.10%** |

Formula-SWCT predicted:

**[0.899803, 0.887323, 0.889689, 0.863769]**

with a best validation accuracy of **37.47%** at epoch **19**.

The Baseline and original S-CT rows will be added after the reference experiment in this notebook is executed.

No conclusions about superiority over S-CT should be drawn until that comparison has been completed.